# 15.7 DeepFM 与 DCN / DeepFM & Deep-Cross Network

**中文**：上一节 Wide & Deep 的软肋是**人工叉乘特征**——要工程师猜哪些组合重要。本节两个模型自动化了这件事，是工业 CTR 排序的主力：
**English**: Wide & Deep's weakness was **manual cross features** — engineers had to guess which combos matter. The two models here automate that and are workhorses of industrial CTR ranking:

- **DeepFM（2017）**：把 Wide&Deep 的"wide 线性塔"换成 **FM**，让二阶交叉**自动学**而非人工设计；FM 部分与 Deep 部分**共享同一套 embedding**。= 自动低阶交叉(FM) + 高阶交叉(Deep)。
- **DCN / Deep & Cross Network（2017）**：设计了**显式的 cross 层**，每加一层多一阶**有界多项式交叉**，比 MLP 更高效、更可解释地建模高阶特征交互。

- **DeepFM (2017)**: replace Wide&Deep's "wide linear tower" with **FM**, so 2nd-order crosses are **learned automatically** rather than hand-designed; the FM and Deep parts **share the same embeddings**. = automatic low-order (FM) + high-order (Deep).
- **DCN / Deep & Cross Network (2017)**: introduces an **explicit cross layer** that adds one polynomial degree of **bounded-degree crossing** per layer — modeling high-order interactions more efficiently and interpretably than a plain MLP.

---

**中文**：**DeepFM 公式**——FM 与 Deep 共享 embedding，输出相加：
**English**: **DeepFM formula** — FM and Deep share embeddings, outputs summed:

$$\hat y = \sigma\big(\,\underbrace{w_0+\textstyle\sum_i w_i x_i + \sum_{i<j}\langle\mathbf v_i,\mathbf v_j\rangle x_i x_j}_{\text{FM：一阶+二阶 / order-1\&2}} + \underbrace{\text{MLP}(\,[\mathbf v_1,\dots,\mathbf v_F]\,)}_{\text{Deep：高阶 / high order}}\,\big)$$

**中文**：关键是 **embedding 共享**——同一个 $\mathbf v_i$ 既喂给 FM 算二阶交叉、又拼起来喂给 MLP 学高阶。这让低阶和高阶信号联合训练、互相校准，且省掉 Wide&Deep 的人工特征工程。
**English**: The key is **shared embeddings** — the same $\mathbf v_i$ feeds the FM (for 2nd-order crosses) and is concatenated into the MLP (for high-order). Low- and high-order signals are jointly trained and mutually calibrated, with zero manual feature engineering.

**中文**：**DCN 的 cross 层**——核心递推式：
**English**: **DCN's cross layer** — the core recurrence:

$$\mathbf x_{l+1} = \mathbf x_0\,(\mathbf x_l^\top \mathbf w_l) + \mathbf b_l + \mathbf x_l$$

**中文**：$\mathbf x_0$ 是初始（embedding 拼接）向量。每经过一层，就把当前特征和**初始特征**再乘一次——所以 $l$ 层 cross 网络能表达**最高 $l{+}1$ 阶**的特征交叉（有界、可控）。末尾的 $+\mathbf x_l$ 是残差连接，利于训练。这比 MLP"隐式学交叉"更显式、参数更省。
**English**: $\mathbf x_0$ is the initial (concatenated embedding) vector. Each layer multiplies the current features by the **original** features once more — so an $l$-layer cross network expresses crosses **up to degree $l{+}1$** (bounded, controllable). The trailing $+\mathbf x_l$ is a residual connection aiding training. This is more explicit and parameter-frugal than letting an MLP learn crosses implicitly.

> 💡 **面试速查 / Interview cheat-sheet（★★★ CTR 排序必考）**
> **中文**：**DeepFM = FM(自动二阶,共享embedding) + Deep(高阶)**，相对 Wide&Deep 的进步是**免人工叉乘**。**DCN = 显式 cross 层(每层+1阶有界多项式) ∥ Deep**。共同点：都用 embedding 表示稀疏特征、都"低阶+高阶并联"。区别：DeepFM 的低阶是 FM(二阶)，DCN 的低阶是 cross 网络(可到高阶、更省参)。**何时用**：海量稀疏类别特征的 CTR/CVR 排序。**诚实点**：MLP 理论上万能，这些结构在 toy 数据上未必胜过纯 deep，其工业价值在于**归纳偏置带来的数据效率 + 稳定性 + 免特征工程**，AUC 增益常只有 0.1%~0.3%——但在十亿级流量上就是大钱。
> **English**: **DeepFM = FM (auto 2nd-order, shared embeddings) + Deep (high-order)**; its advance over Wide&Deep is **no manual crosses**. **DCN = explicit cross layers (each +1 bounded polynomial degree) ∥ Deep**. Common: both embed sparse features and run "low-order + high-order" in parallel. Difference: DeepFM's low-order is FM (2nd), DCN's is the cross net (higher order, more frugal). **When**: CTR/CVR ranking with massive sparse categoricals. **Honest note**: an MLP is a universal approximator, so on toy data these may not beat plain deep; their industrial value is **inductive-bias-driven data efficiency + stability + no feature engineering**, with AUC gains often just 0.1–0.3% — which is big money at billion-scale traffic.


## 实验一：模型对比 / Model comparison on a 2nd + 3rd-order signal

**中文**：我们造一个含**两种交叉信号**的合成数据，正好考验各模型的能力边界：
**English**: We build synthetic data with **two kinds of cross signal**, probing each model's capability boundary:

**中文**：用 5 个二值字段 A,B,C,D,E。
- **二阶信号**：$A\oplus B$（两位奇偶/XOR）——一阶边际为零，是**纯二阶**交叉，**FM 能学、LR 不能**。
- **三阶信号**：$C\oplus D\oplus E$（三位奇偶）——一、二阶边际全为零，是**纯三阶**交叉，**FM 学不到，需要 Deep/Cross**。

**English**: Five binary fields A,B,C,D,E.
- **2nd-order signal**: $A\oplus B$ (2-bit parity/XOR) — zero 1st-order marginal, a **pure 2nd-order** cross that **FM can learn but LR cannot**.
- **3rd-order signal**: $C\oplus D\oplus E$ (3-bit parity) — zero 1st- and 2nd-order marginals, a **pure 3rd-order** cross that **FM cannot learn; it needs Deep/Cross**.


In [ ]:

# ============================================================
# 合成数据 + 五个模型 LR/FM/Deep/DeepFM/DCN（共享 embedding，从零搭）/ synthetic + 5 models
# ============================================================
import numpy as np, torch, torch.nn as nn, matplotlib.pyplot as plt
torch.manual_seed(0); np.random.seed(0)

N, nf = 24000, 5
bits = np.random.randint(0,2,(N,nf))                      # 5 个二值字段 / 5 binary fields
p2 = bits[:,0]^bits[:,1]                                  # 二阶奇偶 A⊕B / 2nd-order
p3 = bits[:,2]^bits[:,3]^bits[:,4]                        # 三阶奇偶 C⊕D⊕E / 3rd-order
logit = 2.2*(2*p2-1) + 2.2*(2*p3-1)                      # 两信号叠加 / both signals
y = (np.random.rand(N) < 1/(1+np.exp(-logit))).astype(np.float32)
F = nf*2                                                  # 每字段 2 类，全局特征数 / 2 cats/field
idx = (np.arange(nf)*2)[None,:] + bits                    # (N,nf) 全局特征下标 / global indices
te=slice(18000,N); yte=y[18000:]; IDX=torch.tensor(idx); Y=torch.tensor(y)

def auc(yy,pp):
    o=np.argsort(pp); r=np.empty(len(pp)); r[o]=np.arange(len(pp))
    a=yy.sum(); b=len(yy)-a; return (r[yy==1].sum()-a*(a-1)/2)/(a*b)

class CTRNet(nn.Module):
    """一个壳实现 5 种模式，全部共享 embedding / one class, five modes, shared embeddings."""
    def __init__(s, mode, k=10, hid=(64,64), ncross=3):
        super().__init__(); s.mode=mode
        s.w=nn.Embedding(F,1); nn.init.zeros_(s.w.weight)       # 一阶线性 / linear
        s.v=nn.Embedding(F,k)                                    # 共享 embedding / shared
        s.b=nn.Parameter(torch.zeros(1)); d=nf*k
        L=[]; din=d
        for h in hid: L+=[nn.Linear(din,h),nn.ReLU()]; din=h
        L+=[nn.Linear(din,1)]; s.deep=nn.Sequential(*L)
        s.cw=nn.ParameterList([nn.Parameter(torch.randn(d)*0.01) for _ in range(ncross)])
        s.cb=nn.ParameterList([nn.Parameter(torch.zeros(d)) for _ in range(ncross)])
        s.cross_out=nn.Linear(d,1)
    def fm(s,e):                                              # FM 二阶项的 O(k) 算法 / FM 2nd-order
        ssum=e.sum(1); sq=(e*e).sum(1); return 0.5*((ssum*ssum).sum(1)-sq.sum(1))
    def forward(s,ix):
        lin=s.b+s.w(ix).squeeze(-1).sum(1); e=s.v(ix)        # 一阶 + embedding / linear + emb
        if s.mode=="lr":     return lin
        if s.mode=="fm":     return lin+s.fm(e)
        if s.mode=="deep":   return lin+s.deep(e.flatten(1)).squeeze(1)
        if s.mode=="deepfm": return lin+s.fm(e)+s.deep(e.flatten(1)).squeeze(1)
        if s.mode=="dcn":                                     # cross 网络 + deep 并联 / cross ∥ deep
            x0=e.flatten(1); xl=x0
            for w,b in zip(s.cw,s.cb): xl=x0*(xl@w).unsqueeze(1)+b+xl
            return lin+s.cross_out(xl).squeeze(1)+s.deep(x0).squeeze(1)

def fit(mode, ep=25, bs=256, lr=5e-3):
    torch.manual_seed(0); m=CTRNet(mode); opt=torch.optim.Adam(m.parameters(),lr); lf=nn.BCEWithLogitsLoss()
    for e in range(ep):
        perm=torch.randperm(18000)
        for i in range(0,18000,bs):
            j=perm[i:i+bs]; opt.zero_grad(); lf(m(IDX[j]),Y[j]).backward(); opt.step()
    m.eval()
    with torch.no_grad(): p=torch.sigmoid(m(IDX[te])).numpy()
    return auc(yte,p)

res={mo:fit(mo) for mo in ["lr","fm","deep","deepfm","dcn"]}
for mo in ["lr","fm","deep","deepfm","dcn"]: print(f"{mo:7}  test AUC = {res[mo]:.4f}")


**中文**：结果与理论完全吻合，且包含一个诚实的要点：
**English**: The results match theory exactly, and include an honest takeaway:

**中文**：
1. **LR ≈ 0.5**：纯奇偶信号没有一阶边际，线性模型完全抓不到。
2. **FM ≈ 0.75**：学到了二阶 $A\oplus B$，但对三阶 $C\oplus D\oplus E$ 无能为力，所以只拿到一半信号。
3. **Deep / DeepFM / DCN ≈ 0.87**：都能同时拿下二阶和三阶——因为 MLP/cross 网络能表达高阶交互。
4. **诚实点：DeepFM、DCN 在这份 toy 数据上并不比纯 Deep 强**。原因很简单——**MLP 是万能逼近器**，信号干净、数据充足时它自己就能学到所有阶的交叉。DeepFM/DCN 的真正价值不在"表达力上限"，而在**归纳偏置**（显式低阶/有界阶交叉 → 数据效率更高、更稳、收敛更快），以及**免人工叉乘**。这一点在工业级稀疏数据上才充分体现，AUC 提升常只有零点几个百分点。

**English**:
1. **LR ≈ 0.5**: pure parity has no 1st-order marginal; a linear model catches nothing.
2. **FM ≈ 0.75**: learns the 2nd-order $A\oplus B$ but is helpless on the 3rd-order $C\oplus D\oplus E$, so it gets only half the signal.
3. **Deep / DeepFM / DCN ≈ 0.87**: all capture both orders — MLP/cross nets can express high-order interactions.
4. **Honest point: DeepFM and DCN do NOT beat plain Deep on this toy data**. The reason is simple — **an MLP is a universal approximator**, so with clean signal and enough data it learns crosses of all orders by itself. DeepFM/DCN's real value is not the expressiveness ceiling but the **inductive bias** (explicit low-/bounded-order crosses → better data efficiency, stability, faster convergence) plus **no manual crosses**. This shows fully on industrial sparse data, where AUC gains are often a fraction of a percent.

**中文**：那 DCN 的 cross 层到底"显式"在哪？下面这个干净的消融实验把它的机制看得清清楚楚。
**English**: So where exactly is DCN's cross layer "explicit"? The clean ablation below makes its mechanism unmistakable.


In [ ]:

# ============================================================
# 实验二：DCN cross 层深度 = 多项式阶数 / DCN depth = polynomial degree
# 中文：单独用 cross 网络(去掉 deep)，喂一个纯三阶奇偶信号，看需要几层才能学会。
#       理论：L 层 cross 网络表达到 L+1 阶；三阶信号需要 L>=2。
# English: cross-only network (no deep) on a PURE 3rd-order parity signal; how many layers are needed?
#          Theory: L cross layers express up to degree L+1; a 3rd-order signal needs L>=2.
# ============================================================
np.random.seed(1); torch.manual_seed(1)
nf3=3; bits3=np.random.randint(0,2,(N,nf3))
p3only=bits3[:,0]^bits3[:,1]^bits3[:,2]                  # 纯三阶信号 / pure 3rd-order
y3=(np.random.rand(N)<1/(1+np.exp(-3.0*(2*p3only-1)))).astype(np.float32)
F3=nf3*2; idx3=(np.arange(nf3)*2)[None,:]+bits3
IDX3=torch.tensor(idx3); Y3=torch.tensor(y3); yte3=y3[18000:]

class CrossOnly(nn.Module):
    def __init__(s,Lc,k=8):
        super().__init__(); s.v=nn.Embedding(F3,k); s.b=nn.Parameter(torch.zeros(1)); d=nf3*k
        s.cw=nn.ParameterList([nn.Parameter(torch.randn(d)*0.05) for _ in range(Lc)])
        s.cb=nn.ParameterList([nn.Parameter(torch.zeros(d)) for _ in range(Lc)])
        s.out=nn.Linear(d,1)
    def forward(s,ix):
        x0=s.v(ix).flatten(1); xl=x0
        for w,b in zip(s.cw,s.cb): xl=x0*(xl@w).unsqueeze(1)+b+xl   # 每层升一阶 / +1 degree per layer
        return s.b+s.out(xl).squeeze(1)
def fit_cross(Lc, ep=30, bs=256, lr=5e-3):
    torch.manual_seed(0); m=CrossOnly(Lc); opt=torch.optim.Adam(m.parameters(),lr); lf=nn.BCEWithLogitsLoss()
    for e in range(ep):
        perm=torch.randperm(18000)
        for i in range(0,18000,bs):
            j=perm[i:i+bs]; opt.zero_grad(); lf(m(IDX3[j]),Y3[j]).backward(); opt.step()
    m.eval()
    with torch.no_grad(): p=torch.sigmoid(m(IDX3[18000:])).numpy()
    return auc(yte3,p)
depth_auc={L:fit_cross(L) for L in [1,2,3,4]}
for L,a in depth_auc.items(): print(f"cross 层数 L={L} (最高阶 {L+1}) : AUC={a:.4f}")


In [ ]:

# ============================================================
# 可视化 / Visualization
# ============================================================
fig,ax=plt.subplots(1,2,figsize=(12,4.3))
# ① 五模型 AUC（2nd+3rd 数据）/ five-model AUC
mo=["lr","fm","deep","deepfm","dcn"]; vals=[res[m] for m in mo]
cols=["#8C8C8C","#C44E52","#55A868","#4C72B0","#8172B3"]
ax[0].bar(mo,vals,color=cols); ax[0].set_ylim(0.45,0.95)
for i,v in enumerate(vals): ax[0].text(i,v+0.008,f"{v:.3f}",ha="center",fontsize=9)
ax[0].axhline(0.5,ls=":",color="k"); ax[0].set_title("2阶+3阶信号：LR<FM<Deep≈DeepFM≈DCN")
ax[0].set_ylabel("test AUC")
# ② DCN cross 深度 vs AUC（纯三阶信号）/ DCN depth vs AUC
Ls=list(depth_auc); av=[depth_auc[L] for L in Ls]
ax[1].plot(Ls,av,"o-",color="#8172B3",ms=9); ax[1].axhline(0.5,ls=":",color="k")
ax[1].set_xticks(Ls); ax[1].set_title("DCN cross 深度 = 可建模阶数 / depth = max degree")
ax[1].set_xlabel("cross 层数 L (最高阶 = L+1)"); ax[1].set_ylabel("AUC (纯3阶信号)")
ax[1].annotate("L=2 起够到 3 阶 → 学会\nneeds degree 3 → L≥2", xy=(2,av[1]), xytext=(2.4,0.7),
               arrowprops=dict(arrowstyle="->"))
plt.tight_layout(); plt.savefig("/tmp/rec07_viz.png",dpi=80); plt.show()
print("纯三阶信号上：L=1 只能部分捕捉，L=2 起跃升 / 3rd-order: L=1 partial, jumps at L>=2:",
      {L:round(depth_auc[L],3) for L in Ls})


**中文**：右图是本节最漂亮的证据——**cross 网络的层数直接对应它能建模的多项式阶数**：
**English**: The right plot is this section's cleanest evidence — **the cross network's depth directly equals the polynomial degree it can model**:

**中文**：纯三阶奇偶信号下，**L=1（最高二阶）只能部分捕捉**（AUC ~0.67，够不到三阶），**L=2（最高三阶）骤然跃升**（AUC ~0.95），再加层只是饱和。阶数一旦够到信号所需的三阶，性能就阶跃式上升。这就是 DCN 的卖点：用**很少的参数**、以**可控的阶数**显式建模高阶交叉——不像 MLP 那样"黑箱地、隐式地"逼近，也不像 FM 那样被锁死在二阶。
**English**: On a pure 3rd-order parity signal, **L=1 (max degree 2) only partially captures it** (AUC ~0.67, can't reach 3rd order), while **L=2 (max degree 3) jumps sharply** (AUC ~0.95), and more layers just saturate. Once the degree reaches the signal's required 3rd order, performance steps up. That is DCN's selling point: explicitly model high-order crosses with **few parameters** at a **controllable degree** — unlike the MLP's black-box implicit approximation, and unlike FM being locked at 2nd order.

> 💼 **实战视角 / Practical angle**
> **中文**：工业 CTR 排序的主流就是"**embedding + 低阶交叉(FM/Cross) + 高阶(Deep)**"这套范式：DeepFM、DCN、DCN-V2、xDeepFM、AutoInt… 都是它的变体。选型经验：① 特征以稀疏类别为主、要免特征工程 → DeepFM 起步；② 想显式控制交叉阶数、参数敏感 → DCN/DCN-V2；③ **永远和纯 Deep 基线比**，很多时候增益很小，要权衡复杂度。面试金句：*"这些模型的本质都是把'低阶记忆'和'高阶泛化'用共享 embedding 联合起来；MLP 万能但归纳偏置能换来数据效率。"*
> **English**: Industrial CTR ranking is dominated by the paradigm "**embeddings + low-order crossing (FM/Cross) + high-order (Deep)**": DeepFM, DCN, DCN-V2, xDeepFM, AutoInt… are all variants. Selection tips: ① mostly sparse categoricals, want no feature engineering → start with DeepFM; ② want explicit control of cross degree, parameter-sensitive → DCN/DCN-V2; ③ **always compare to a plain Deep baseline** — gains are often small, weigh the complexity. Interview line: *"They all unify 'low-order memorization' and 'high-order generalization' via shared embeddings; the MLP is universal, but inductive bias buys data efficiency."*

---
### 小结 / Summary
- **中文**：DeepFM = FM(自动二阶,共享embedding) + Deep(高阶)，免人工叉乘；DCN = 显式 cross 层(每层+1阶) ∥ Deep。
- **English**: DeepFM = FM (auto 2nd, shared embeddings) + Deep (high-order), no manual crosses; DCN = explicit cross layers (+1 degree each) ∥ Deep.
- **中文**：合成实验 LR<FM<Deep≈DeepFM≈DCN——MLP 万能，结构的价值在归纳偏置/数据效率/免特征工程，非表达力上限。
- **English**: Synthetic: LR<FM<Deep≈DeepFM≈DCN — the MLP is universal; these structures add inductive bias / data efficiency / no feature engineering, not a higher ceiling.
- **中文**：DCN cross 层数 = 可建模的多项式阶数（L 层→ L+1 阶），这是它显式、省参、可控的关键。
- **English**: DCN cross depth = modelable polynomial degree (L layers → degree L+1) — its explicit, frugal, controllable core.
